# Phát hiện bất thường chuỗi thời gian — theo notebook tham chiếu

Notebook tái lập hướng phân tích bạn cung cấp: dữ liệu CPU EC2 đầy đủ của NAB, biểu đồ nhãn thật, phân phối giá trị, baseline MAD, Isolation Forest và LOF. Các phép tính dùng chung `src.notebook_reference` với BE; không chép lại phần huấn luyện ở đây.

**Phạm vi cần phân biệt:** MAD mô tả toàn bộ 4.032 điểm và dùng thống kê của chính toàn bộ dữ liệu. Isolation Forest/LOF học 3.550 điểm đầu, dự đoán 482 điểm cuối. Đây là hai phạm vi đánh giá khác nhau; không xếp hạng trực tiếp các chỉ số đó. Nhãn là hai mốc từng điểm trong NAB, không phải điểm chấm NAB theo cửa sổ.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# Hỗ trợ: workspace root, DACN/, ml-anomaly-detection/ hoặc notebooks/.
cwd = Path.cwd().resolve()
candidates = []
for base in (cwd, *cwd.parents):
    candidates.extend((base, base / "ml-anomaly-detection", base / "DACN" / "ml-anomaly-detection"))
ROOT = next((candidate for candidate in candidates
             if (candidate / "src" / "notebook_reference.py").is_file()
             and (candidate / "data" / "ec2_cpu_utilization.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy dự án và data/ec2_cpu_utilization.csv đầy đủ.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.notebook_reference import run_notebook_reference

plt.rcParams["figure.figsize"] = (11, 4)
print("Thư mục dự án:", ROOT)


## 1. Đọc dữ liệu và hai nhãn gốc

Nguồn: [NAB EC2 CPU](https://github.com/numenta/NAB/blob/master/data/realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv), [nhãn từng điểm](https://github.com/numenta/NAB/blob/master/labels/combined_labels.json). Dữ liệu có 4.032 điểm cách nhau 5 phút. Giữ nguyên thang đo gốc, không tự đổi đơn vị. Bản sao dữ liệu, nhãn, mã commit và mã băm được ghi ở `data/nab_provenance.json`; giấy phép đi kèm `data/NAB_LICENSE.txt`.

Hai mốc chú giải là **2014-02-26 22:05:00** và **2014-02-27 17:15:00**. Quy ước `1` = bình thường theo chú giải, `-1` = bất thường. Nhãn chỉ dùng để đánh giá.


In [ ]:
source = ROOT / "data" / "ec2_cpu_utilization.csv"
run = run_notebook_reference(source)
df = run.frame.copy()
baseline = run.metadata["reference_baseline"]
baseline_predictions = pd.DataFrame(baseline["predictions"])
baseline_predictions["timestamp"] = pd.to_datetime(baseline_predictions["timestamp"], utc=True)
split = run.metadata["split_index"]
train, test = df.iloc[:split], df.iloc[split:]
anomalies_timestamp = [
    "2014-02-26 22:05:00",
    "2014-02-27 17:15:00",
]
anomaly_df = df.loc[df["is_anomaly"] == -1]
inlier_df = df.loc[df["is_anomaly"] == 1]
print(f"Tổng: {len(df)} điểm; từ {df['timestamp'].min()} đến {df['timestamp'].max()}")
print("Hai mốc nhãn:", anomalies_timestamp)
print(f"Train: {len(train)}; test: {len(test)}")
print("Bất thường có nhãn trong train/test:",
      int((train["is_anomaly"] == -1).sum()),
      int((test["is_anomaly"] == -1).sum()))
df.head()


In [ ]:
fig, ax = plt.subplots()
ax.scatter(inlier_df["timestamp"], inlier_df["value"], color="#2563eb", s=3, label="Điểm còn lại")
ax.scatter(anomaly_df["timestamp"], anomaly_df["value"], color="#dc2626", s=45, label="Hai nhãn NAB", zorder=3)
ax.axvline(test["timestamp"].iloc[0], color="black", linestyle="--", label="Bắt đầu 482 điểm test")
ax.set(title="CPU EC2 — toàn bộ 4.032 điểm và nhãn gốc", xlabel="Thời gian", ylabel="Giá trị gốc")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## 2. Baseline: median absolute deviation (MAD)

Theo notebook tham chiếu, median và MAD được tính từ **toàn bộ 4.032 điểm**. Robust Z-score là `0.6745 × (value − median) / MAD`; đánh dấu bất thường khi `|z| ≥ 3.5`, gồm cả tăng và giảm. Đường đứng ở biểu đồ dưới dùng median tính thực tế, không ghi cứng số 0.134.

Đây là baseline mô tả dữ liệu đã quan sát. Nó dùng cả phần test để tính thống kê, nên **không phải thử nghiệm chỉ học quá khứ**.


In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(data=df, x="value", ax=ax, fill=True, color="#2563eb")
ax.axvline(baseline["median"], color="black", linestyle="--", label=f"Median = {baseline['median']:.6g}")
ax.set(title="Phân phối toàn bộ giá trị CPU", xlabel="Giá trị gốc", ylabel="Mật độ")
ax.legend()
fig.tight_layout()
plt.show()

print("Median:", baseline["median"])
print("MAD:", baseline["mad"])
print("Ngưỡng |robust Z-score|:", baseline["threshold"])
baseline_predictions.head()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    np.asarray(baseline["confusion_matrix"]),
    display_labels=["Bình thường", "Bất thường"],
).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set(title="MAD: đánh giá toàn bộ 4.032 điểm")
fig.tight_layout()
plt.show()
print("MAD toàn bộ dữ liệu:", baseline["metrics"])


## 3. Isolation Forest — học 3.550, kiểm tra 482 điểm

Giữ đúng hướng tham chiếu: chỉ dùng cột `value` gốc, không StandardScaler và không thêm đặc trưng thời gian. `contamination = 1/3550`, `random_state = 42`; không chỉnh tham số bằng nhãn test. Một nhãn bất thường nằm trong train, một nhãn nằm trong test.


In [ ]:
contamination = run.metadata["contamination"]
comparison = run.predictions.copy()
print("contamination =", contamination)
print("Cấu hình Isolation Forest:", run.metadata["parameters"]["isolation_forest"])

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    np.asarray(run.metadata["confusion_matrices"]["isolation_forest"]),
    display_labels=["Bình thường", "Bất thường"],
).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set(title="Isolation Forest: 482 điểm test")
fig.tight_layout()
plt.show()
print("Isolation Forest trên test:", run.metrics["isolation_forest"])


## 4. Local Outlier Factor (LOF)

LOF dùng cùng train/test, giá trị gốc và `contamination` như Isolation Forest. `novelty=True` cho phép học dữ liệu quá khứ rồi dự đoán những điểm test chưa dùng để fit.


In [ ]:
print("Cấu hình LOF:", run.metadata["parameters"]["lof"])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    np.asarray(run.metadata["confusion_matrices"]["lof"]),
    display_labels=["Bình thường", "Bất thường"],
).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set(title="LOF: 482 điểm test")
fig.tight_layout()
plt.show()
print("LOF trên test:", run.metrics["lof"])


## 5. So sánh có ghi rõ phạm vi

Bảng đầu giữ cách đánh giá của notebook tham chiếu: MAD trên 4.032 điểm (2 nhãn), IF/LOF trên 482 điểm test (1 nhãn). Không coi đây là bảng xếp hạng công bằng giữa ba thuật toán.

Bảng thứ hai chỉ xét **cùng 482 điểm test**. Dù dùng cùng số điểm chấm, MAD vẫn lấy median/MAD từ toàn bộ dữ liệu; không được gọi là so sánh ba mô hình đều chỉ fit train. Muốn so sánh chỉ học quá khứ, dùng chế độ CSV tùy chỉnh của ứng dụng, nơi tất cả mô hình fit trên train.


In [ ]:
reference_rows = [{"thuật_toán": "MAD (toàn bộ)", "số_điểm_đánh_giá": len(df),
                   "bất_thường_có_nhãn": len(anomaly_df), **baseline["metrics"]}]
for name in ("isolation_forest", "lof"):
    reference_rows.append({"thuật_toán": name, "số_điểm_đánh_giá": len(test),
                           "bất_thường_có_nhãn": int((test["is_anomaly"] == -1).sum()),
                           **run.metrics[name]})
reference_metrics_table = pd.DataFrame(reference_rows).set_index("thuật_toán")
print("PHẠM VI THEO NOTEBOOK THAM CHIẾU:")
print(reference_metrics_table.to_string())

test_annotation_count = int((comparison["is_anomaly"] == -1).sum())
quality_supported = test_annotation_count > 0
metrics_table = pd.DataFrame.from_dict(run.metrics, orient="index")
print("\nCÙNG 482 ĐIỂM TEST (MAD vẫn dùng thống kê toàn bộ dữ liệu):")
print(metrics_table.to_string())
print("\nNhãn thưa: test chỉ có một mốc bất thường. Chưa đủ suy rộng chất lượng thực tế.")
print("Chỉ số từng điểm; không phải cơ chế chấm điểm NAB theo cửa sổ.")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, name in zip(axes, run.metadata["model_names"]):
    ax.plot(df["timestamp"], df["value"], color="#cbd5e1", linewidth=0.7, label="Toàn bộ dữ liệu")
    ax.plot(comparison["timestamp"], comparison["value"], color="#2563eb", linewidth=0.9, label="482 điểm test")
    alerts = comparison.loc[comparison[f"{name}_prediction"] == -1]
    truth = comparison.loc[comparison["is_anomaly"] == -1]
    ax.scatter(alerts["timestamp"], alerts["value"], color="#dc2626", s=18, label="Cảnh báo mô hình", zorder=3)
    ax.scatter(truth["timestamp"], truth["value"], color="black", marker="x", s=70, label="Nhãn NAB trong test", zorder=4)
    ax.axvline(test["timestamp"].iloc[0], color="black", linestyle="--")
    ax.set(title=name, ylabel="Giá trị gốc")
    ax.legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## 6. Bằng chứng để đưa vào báo cáo

Chạy `python scripts/run_notebook_reference.py` bằng môi trường dự án từ thư mục workspace để tạo ảnh, ma trận nhầm lẫn, CSV dự đoán và JSON kết quả trong `reports/notebook_reference/`. Script gọi lại đúng hàm phân tích ở trên; không nhập sẵn kết quả mong đợi.

Khi trình bày: chỉ nguồn dữ liệu và hai nhãn gốc; nêu mốc chia 3.550/482; chỉ điểm gắn cờ; đọc ma trận nhầm lẫn theo hàng nhãn thật/cột dự đoán, thứ tự `[1, -1]` tương ứng `[[TN, FP], [FN, TP]]`; giải thích cảnh báo nhầm và bỏ sót. Kết quả chỉ có ý nghĩa trong phạm vi dữ liệu/nhãn này. Dữ liệu CPU EC2 chưa chứng minh việc giám sát hệ quản trị cơ sở dữ liệu.
